# ⛓️ Blockchain from zero — the ledger nobody owns

In `05_the_walking_skeleton.ipynb` the "blockchain" was a Python dict named `FakeChain`,
and you watched it enforce single-use offers and atomic settlement — on cardboard. This
chapter replaces the cardboard with the real thing: a real (if miniature) Ethereum chain
running on your machine, real keys, real transactions that cost gas, and the real
`Settlement.sol` — read line by line, in Solidity, before `07` starts driving it.

**What you'll be able to do after this notebook**

- explain account / balance / transaction / block / gas to a friend, and say precisely what "trustless" buys Ada and Bell
- derive an address from a private key, and prove the fixture addresses were key-derived all along
- launch a disposable Anvil chain pinned to story time, then read and write it with web3.py — knowing which of the two is free
- read Solidity you've never seen: structs, mappings, events, custom errors, inheritance from OpenZeppelin
- walk `Settlement.sol` end to end and point at the exact lines that make tickets single-mint, revocable-not-burnable, and 404-proof
- show, with `forge inspect`, the two ledgers living side by side in the contract's storage slots

**You need:** notebooks 00–05 (the story, Python constructs, pydantic shapes, Protocols,
the canonical example, the skeleton's fakes).
**Infra:** Foundry (`anvil` + `forge`) with built artifacts (`forge build --root contracts`).
Without it, every chain cell prints a polite skip line and the reading course still works.

**Estimated time:** ~75 minutes.

> **How to run:** ▶▶ Run All, or step through with Shift+Enter. Each markdown cell tells
> you what to look for in the code cell below it.

## 1. The trust problem — why a ledger nobody controls

Rewind to the story's first scene. Ada's agent needs 50 Mbps; Bell's agent sells it. They
are strangers — no shared bank to hold the money, no court to sue in when one side cheats,
and neither will move first ("you pay, then I provision" vs "you provision, then I pay").
Every marketplace in history solved this with a **trusted middleman**. The blockchain's
one trick is to replace the middleman with a **ledger no single party controls**: a shared
record of who-owns-what that Ada and Bell can both write to, both read, and neither can
falsify. "Trustless" doesn't mean nobody trusts anything — it means *you don't have to
trust your counterparty*, only the machinery, which is public and checkable.

The five words you need (everything else builds on these):

| word | meaning |
|---|---|
| **account** | an entry in the ledger: an address with a balance (and, we'll see, sometimes code) |
| **balance** | how much of the chain's native money (ETH here) an account holds |
| **transaction** | a *signed* instruction that changes the ledger — "move 1 wei from me to Bell" |
| **block** | a numbered batch of transactions, stamped with a timestamp; the chain is the list of blocks |
| **gas** | the fee per unit of computation a transaction causes — you pay for the work you create |

Two honest sentences about **consensus**: on a public chain, thousands of independent
nodes run every transaction and agree on each block, and that agreement is what makes the
ledger hard to falsify. Our chain is **Anvil — a one-node simulator on your laptop** — so
"consensus" here is one process agreeing with itself; the *programming model* is
identical, the trust-distribution is theatre.

First, the stage crew: locate the repo, meet the cast, and check which props this machine
has. Watch the two flags — they decide whether the live-chain cells below run or politely
skip.

In [ ]:
from pathlib import Path

import a2a_interfaces.fixtures as fx
from chainmcp.testing import anvil_available, artifacts_available

# find the repo root from wherever this notebook runs (cwd is e2e/notebooks/course/)
ROOT = next(p for p in [Path.cwd(), *Path.cwd().resolve().parents] if (p / ".git").exists())
assert (ROOT / "contracts" / "src" / "Settlement.sol").exists()

ARTIFACTS_OK = artifacts_available()            # forge build outputs in contracts/out/
CHAIN_OK = anvil_available() and ARTIFACTS_OK   # ... plus the anvil binary on PATH
SKIP = ("skipped: needs anvil + forge artifacts — install Foundry "
        "(curl -L https://foundry.paradigm.xyz | bash && foundryup), "
        "then run: forge build --root contracts")

print("repo root       →", ROOT.name)
print("the cast        →  Ada", fx.ADA[:10] + "…", "· Bell", fx.BELL[:10] + "…")
print("anvil on PATH   →", "✓" if anvil_available() else "✗")
print("forge artifacts →", "✓" if ARTIFACTS_OK else "✗")
if not CHAIN_OK:
    print(SKIP)

## 2. The magic moment: your identity was a number all along

Since `04` you've treated `fx.ADA` — `0xf39F…2266` — as "Ada's address", a given. It
never was. A blockchain **account** is nothing but a key pair: a **private key** (a
256-bit random number you keep secret), from which a public key is computed, from which
the **address** is computed (the last 20 bytes of the public key's keccak hash — a
*fingerprint*, in `04`'s sense). No registration, no signup form, no database row:
*knowing the number IS being Ada.*

How do you prove you know it without revealing it? You **sign**: a signature is a
computation only the key-holder can produce and anyone can verify — proof of key
possession. That's the trick behind every transaction in §4 and every offer Bell signs in
`07`, where signing gets its deep dive.

The next cell performs the reveal: feed the well-known dev key into `eth_account` (the
Python library for exactly this math) and watch Ada's address fall out.

In [ ]:
from eth_account import Account

from chainmcp.testing import ANVIL_KEYS   # the four well-known dev keys — see the warning below

ada = Account.from_key(ANVIL_KEYS["ada"])

print("private key  →", ANVIL_KEYS["ada"])
print("address      →", ada.address)
print("fixtures.ADA →", fx.ADA)
assert ada.address == fx.ADA   # the address you have used since 04 falls out of the key
print("\nAda's 'identity' was this number all along ✓")

> **Warning — these keys are public.** `ANVIL_KEYS` are Anvil's well-known dev-account
> keys, printed in Foundry's docs and preloaded on every Anvil chain on earth. They are
> stage props: anything of real value sent to their addresses on a public chain is gone
> in seconds. This is exactly why **hard rule 2** exists — *private keys live only in
> `chainmcp`* — and the repo applies it even to these fake keys: the literal strings live
> in `chainmcp/src/chainmcp/testing.py` and nowhere else. The pattern is the point: a key
> literal that "doesn't matter" today becomes a habit tomorrow.

The whole cast, next: all four dev keys, each just a very large number, each collapsing
to an address the story has used all along.

In [ ]:
for name, key in ANVIL_KEYS.items():
    address = Account.from_key(key).address
    print(f"{name:8} {address}   ← from a {int(key, 16).bit_length()}-bit secret number")

assert Account.from_key(ANVIL_KEYS["bell"]).address == fx.BELL
print("\nBell checks out too ✓ — an account is a key pair, nothing more")

**✏️ Your turn 2.1 — unmask Mallory**

`05` introduced Mallory, the TOK-broke freeloader, only as an address imported from
`e2e.skeleton.worlds`. Derive her address from `ANVIL_KEYS["mallory"]` and confirm the
skeleton was using anvil account #3 all along. Success: the un-commented assert passes.

In [ ]:
from e2e.skeleton.worlds import MALLORY   # the freeloader's address, exactly as 05 used it

mallory_address = None
# TODO: derive it — mallory_address = Account.from_key(...).address

print("derived  :", mallory_address)
print("expected :", MALLORY)
# assert mallory_address == MALLORY   # un-comment once derived

<details><summary>✅ Solution 2.1 — peek only after trying</summary>

```python
mallory_address = Account.from_key(ANVIL_KEYS["mallory"]).address
assert mallory_address == MALLORY
```

One line: the skeleton's `MALLORY` constant was derived from dev key #3 — every identity
in this project bottoms out in a key.
</details>

## 3. A disposable world: read `launch_anvil` before running it

**Anvil** is Foundry's local chain: one process that behaves like Ethereum — accounts,
blocks, gas, contracts — but lives on your laptop, starts in under a second, pre-funds
ten dev accounts with 10,000 ETH each, and dies when you're done. Perfect for a course:
we can create and destroy *worlds*.

The repo wraps the launch in `chainmcp.testing.launch_anvil`. House ritual: **read the
machine before switching it on.** Three pieces, smallest first — the handle it returns.
`AnvilChain` is a `@dataclass` (`01`: the decorator that writes `__init__`/`__repr__` for
you from the field list). Note what a "world" amounts to: a process, a URL, and a dict of
deployed addresses — plus `increase_time`, the time-warp lever only tests and labs may
pull.

In [ ]:
import inspect

import chainmcp.testing as testing

print(inspect.getsource(testing.AnvilChain))

Next helper down: where does the port number come from? `_free_port` (leading `_` =
internal; we peek to learn) uses the classic trick — bind a socket to port 0 and the OS
picks a spare port for you. The `with` statement (`01`: a context manager) guarantees the
socket closes, releasing the port for anvil to grab a moment later. This is why two
notebooks can run side by side without colliding — and why you must never hardcode an RPC
port: always read it off the handle.

In [ ]:
print(inspect.getsource(testing._free_port))   # _private: peeking to learn

print("two picks:", testing._free_port(), "and", testing._free_port(), "← the OS hands out spare ports")

And the launcher itself. Read it top to bottom and collect four decisions:

1. **`subprocess.Popen`** starts `anvil` as a separate program (`01` met threads; this is
   the heavier sibling — a whole other process), output discarded.
2. **The pinned timestamp** — `--timestamp` sets the new world's clock to *whatever we
   choose*. Chain time is story time (ADR-004): every chain notebook pins the clock near
   13:32 on 15 Sep 2025, so the canonical offer is alive and Ada's 14:00 window is ahead.
3. **Readiness polling** — a deadline loop retries `chain_id` until anvil answers (up to
   10 s), because `Popen` returns before anvil is ready to serve.
4. **Deploy order is load-bearing** — MockTOK first, from account #0 (Ada). A fresh
   chain's contract addresses depend only on (deployer, nonce), so deploy #1 always lands
   at the exact address `fixtures.MOCK_TOK` promised back in `04`. Determinism, not luck.

In [ ]:
print(inspect.getsource(testing.launch_anvil))

`launch_anvil` mirrors the *real* deploy script, `contracts/script/Deploy.s.sol` — a
Foundry **Script**: Solidity that runs off-chain (never deployed itself) with `vm.*`
superpowers, called cheatcodes. `vm.startBroadcast()` turns each `new Contract()` into a
real signed transaction; afterwards the script writes `contracts/deployments/anvil.json`
— the one artifact every Python package reads to find the chain (`just deploy-local` runs
this). Same two deploys, same order, same reason:

In [ ]:
print((ROOT / "contracts" / "script" / "Deploy.s.sol").read_text())

Now switch the world on. `STORY_TIME` is 13:32 — 28 minutes before Ada's window opens at
14:00, 48 before the quote expires at 14:20 (both from `fx`, as always). Watch for two
things: the RPC URL's port differs every run (that's `_free_port` working), and the
deployed MockTOK address equals `fx.MOCK_TOK` **exactly** — the fixture's promise, kept
by deploy order.

In [ ]:
anvil = None
STORY_TIME = fx.WINDOW.start - 1680     # 13:32 — 28 min before Ada's 14:00 window

if not CHAIN_OK:
    print(SKIP)
else:
    anvil = testing.launch_anvil(timestamp=STORY_TIME)
    print("rpc        →", anvil.rpc_url, "  ← fresh port, every run")
    print("deployment →", anvil.deployment)
    assert anvil.deployment["MockTOK"] == fx.MOCK_TOK   # the fixture's promise, kept
    print("\nMockTOK landed exactly where fixtures.MOCK_TOK promised ✓")

**✏️ Your turn 3.1 — a second world, same addresses**

Determinism is a strong claim — test it. Launch a *second* anvil pinned to a completely
different time, `fx.WINDOW.end` (16:00, the moment Ada's window closes), and compare its
`deployment` dict to the first world's. Success: identical addresses on both chains —
then `stop()` your world (never orphan a chain).

In [ ]:
if not CHAIN_OK:
    print(SKIP)
else:
    two = None
    # TODO: two = testing.launch_anvil(timestamp=fx.WINDOW.end)
    if two is None:
        print("(scaffold unchanged — launch the second world, then rerun)")
    else:
        print("second world →", two.deployment)
        print("identical?   →", two.deployment == anvil.deployment)
        two.stop()                                    # never orphan a chain
        # assert two.deployment == anvil.deployment   # un-comment: determinism, proven

<details><summary>✅ Solution 3.1 — peek only after trying</summary>

```python
two = testing.launch_anvil(timestamp=fx.WINDOW.end)
print(two.deployment == anvil.deployment)   # True
two.stop()
assert two.deployment == anvil.deployment
```

Different clock, same (deployer, nonce) history → byte-identical addresses; time has no
say in where contracts land.
</details>

## 4. First contact: web3.py

Anvil speaks **RPC** (remote procedure call): you POST a named request over HTTP —
`eth_chainId`, `eth_getBlockByNumber` — and get JSON back. **web3.py** is Python's client
for that protocol: `Web3(Web3.HTTPProvider(url))` gives you a `w3` handle, and everything
chain-shaped lives under `w3.eth`.

First reads — and one assert that matters more than it looks: the block **timestamp is
the story clock**. The genesis block (block 0, the chain's birth certificate) is stamped
*exactly* `STORY_TIME`; the two deploy blocks were mined within a wall-clock second of
it. There is no other clock — when the controller later asks "is the window open?",
**this** number answers (ADR-004).

In [ ]:
import datetime

from web3 import Web3

if not CHAIN_OK:
    print(SKIP)
else:
    w3 = Web3(Web3.HTTPProvider(anvil.rpc_url))
    assert w3.is_connected() and w3.eth.chain_id == 31337     # anvil's well-known chain id

    genesis, latest = w3.eth.get_block(0), w3.eth.get_block("latest")
    print("chain_id     →", w3.eth.chain_id)
    print("latest block →", latest["number"], "  (genesis + two deploy transactions)")
    assert latest["number"] == 2
    assert genesis["timestamp"] == STORY_TIME                 # the pin, byte-exact
    assert 0 <= latest["timestamp"] - STORY_TIME <= 2         # deploys mined a wall-moment later
    when = datetime.datetime.fromtimestamp(latest["timestamp"], datetime.UTC)
    print("chain time   →", latest["timestamp"], "=", when.strftime("%H:%M:%S %d %b %Y"), "— story time ✓")
    assert latest["timestamp"] < fx.WINDOW.start              # we sit before Ada's window opens

Money next — and a disambiguation that saves real grief. `get_balance` returns **ETH**,
the chain's *native* money: it pre-exists all contracts, and it is what pays for gas. It
is **not TOK** — the story's payment token is an ERC-20 *contract* we meet in §5, a
ledger-within-the-ledger. Two currencies, two ledgers. Units: 1 ETH = 10¹⁸ **wei**, and
balances are integers of wei — the same integer-money discipline `04` drilled for TOK.

Watch for the asymmetry: Bell holds exactly 10,000 ETH (his pre-fund, untouched), Ada
slightly less — she signed the two deploy transactions inside `launch_anvil` and paid
their gas.

In [ ]:
if not CHAIN_OK:
    print(SKIP)
else:
    for name, address in [("Ada", fx.ADA), ("Bell", fx.BELL)]:
        wei = w3.eth.get_balance(address)
        print(f"{name:5} {wei:>26} wei  =  {w3.from_wei(wei, 'ether')} ETH")
    assert w3.eth.get_balance(fx.BELL) == 10_000 * 10**18    # untouched pre-fund
    assert w3.eth.get_balance(fx.ADA) < 10_000 * 10**18      # Ada paid the deploy gas
    print("\nAda is short a little ETH — deploying the two contracts cost her gas")

Time to change the world: send **1 wei** from Ada to Bell. A transaction is a dict of
fields, signed, submitted, mined:

- `nonce` — the account's transaction counter (0, 1, 2, …). It orders your transactions
  and makes each land at most once. (A *cousin* of the auth nonce you'll meet in `08` —
  same anti-replay idea, different mechanism.)
- `gas: 21_000` — the fixed price of a plain transfer, the cheapest thing a chain does.
- `chainId` — baked into the signature so a transaction meant for this chain can't be
  replayed onto another.
- then `Account.sign_transaction(tx, key)` — §2's proof-of-key-possession, doing its job.

Watch three outcomes: `status: 1` (success), `gasUsed` exactly 21000, and the block
number ticking up by one. **The world's state changed because Ada signed an instruction.**

In [ ]:
if not CHAIN_OK:
    print(SKIP)
else:
    bell_before, block_before = w3.eth.get_balance(fx.BELL), w3.eth.block_number
    tx = {
        "to": fx.BELL, "value": 1,                        # 1 wei — the smallest possible payment
        "gas": 21_000,                                    # fixed cost of a plain transfer
        "gasPrice": w3.eth.gas_price,
        "nonce": w3.eth.get_transaction_count(fx.ADA),    # Ada's transaction counter
        "chainId": w3.eth.chain_id,                       # pins the signature to THIS chain
    }
    signed_tx = Account.sign_transaction(tx, ANVIL_KEYS["ada"])
    tx_hash = w3.eth.send_raw_transaction(signed_tx.raw_transaction)
    receipt = w3.eth.wait_for_transaction_receipt(tx_hash)

    print("mined in block →", receipt["blockNumber"], "  (was", str(block_before) + ") · status:", receipt["status"])
    print("gas used       →", receipt["gasUsed"])
    assert receipt["status"] == 1 and receipt["gasUsed"] == 21_000
    assert receipt["blockNumber"] == block_before + 1     # the world grew by one block
    assert w3.eth.get_balance(fx.BELL) - bell_before == 1
    print("Bell +1 wei ✓ — a signed instruction changed the world's state")

The distinction that organizes everything downstream: **call vs transaction**.

| | call | transaction |
|---|---|---|
| what it is | a *question* to your node | a signed *event* in history |
| cost | free | gas |
| signature | none needed | required |
| state afterwards | unchanged | changed — a block is mined |

Every read so far (`get_block`, `get_balance`) was a call; the 1-wei send was a
transaction. Hold onto this: the controller in `08` only ever **calls** — it verifies
ownership and reads terms for free, thousands of times if it likes, and never signs
anything (rule 2, seen from the other side). Proof by counting blocks:

In [ ]:
if not CHAIN_OK:
    print(SKIP)
else:
    before = w3.eth.block_number
    for _ in range(50):
        w3.eth.get_balance(fx.ADA)
    assert w3.eth.block_number == before
    print("50 reads → 0 new blocks, 0 gas ✓  (calls are questions, not events)")

**✏️ Your turn 4.1 — predict, then pay Carol**

The scaffold sends **5 wei** from Ada to Carol. Before you run it, write your predictions
into the `TODO` comment: what will Carol's balance delta be, and what block number will
the receipt show (you just saw the current one)? Then run, compare, and un-comment the
asserts. Success: both asserts pass and your predictions match.

In [ ]:
if not CHAIN_OK:
    print(SKIP)
else:
    CAROL = Account.from_key(ANVIL_KEYS["carol"]).address
    # TODO — predict BEFORE running:  carol's delta = ?   receipt block number = ?
    carol_before, block_before = w3.eth.get_balance(CAROL), w3.eth.block_number
    tx = {"to": CAROL, "value": 5, "gas": 21_000, "gasPrice": w3.eth.gas_price,
          "nonce": w3.eth.get_transaction_count(fx.ADA), "chainId": w3.eth.chain_id}
    signed_tx = Account.sign_transaction(tx, ANVIL_KEYS["ada"])
    receipt = w3.eth.wait_for_transaction_receipt(w3.eth.send_raw_transaction(signed_tx.raw_transaction))
    print("carol delta →", w3.eth.get_balance(CAROL) - carol_before)
    print("block       →", receipt["blockNumber"], "  (was", str(block_before) + ")")
    # assert w3.eth.get_balance(CAROL) - carol_before == 5
    # assert receipt["blockNumber"] == block_before + 1

<details><summary>✅ Solution 4.1 — peek only after trying</summary>

```python
# delta = 5 (value moves exactly as written); block = block_before + 1
assert w3.eth.get_balance(CAROL) - carol_before == 5
assert receipt["blockNumber"] == block_before + 1
```

Anvil mines exactly one block per transaction, and value moves to the wei — no rounding,
no fees taken from the recipient (gas is paid by the *sender*, on top).
</details>

## 5. A contract is code + storage living at an address

So far, an address meant a key pair. The second kind of account is the big idea: a
**smart contract** is a program *deployed at an address*. Its code is public, its
**storage** (its variables) is public, and the only way to change that storage is to send
a transaction to one of its functions. No operator, no server, no "admin fixes the DB by
hand" — code and state, living in the ledger.

To talk to code you need its menu. The **ABI** (Application Binary Interface) is exactly
that: a JSON list describing every function (name, typed inputs, outputs), event, and
error — everything a client needs to encode a call. Ours come from `forge build`'s
artifacts under `contracts/out/`, loaded by `chainmcp.artifacts`. Read the locator first —
the walk-up trick that finds `contracts/` from any working directory (notebook, pytest,
MCP server), no configuration needed:

In [ ]:
import chainmcp.artifacts as artifacts

print(inspect.getsource(artifacts.find_contracts_dir))
print("found →", artifacts.find_contracts_dir())

Now load the real menu for `A2ASettlement` and read one row properly: `fulfill` — the
function this whole project revolves around. Watch the types: it takes a `tuple` (the
12-field `Offer` you dissected value-by-value in `04` — recognize the names) plus `bytes`
(the provider's signature), and returns a `uint256` — the freshly minted ticket id.

In [ ]:
from collections import Counter

if not ARTIFACTS_OK:
    print(SKIP)
else:
    abi = artifacts.load_abi("A2ASettlement")
    print("menu size:", len(abi), "entries →", dict(Counter(e.get("type") for e in abi)))
    assert {e.get("type") for e in abi} == {"function", "event", "error", "constructor"}

    fulfill = next(e for e in abi if e.get("name") == "fulfill")
    print("\nfulfill's row on the menu:")
    print("  inputs  →", [(i["name"], i["type"]) for i in fulfill["inputs"]])
    print("  output  →", fulfill["outputs"][0]["type"])
    print("  tuple fields →", [c["name"] for c in fulfill["inputs"][0]["components"]])
    assert [i["type"] for i in fulfill["inputs"]] == ["tuple", "bytes"]   # an Offer + a signature
    assert fulfill["outputs"][0]["type"] == "uint256"                     # → the new ticket id

Before *using* the payment token, read it — all twenty lines. **ERC-20** is the standard
menu for **fungible** tokens (every unit identical and interchangeable, like money in a
bank column): `balanceOf`, `transfer`, `approve`. `MockTOK is ERC20` inherits the entire
audited OpenZeppelin implementation and adds exactly one function: an **open faucet** —
anyone may mint. Deliberate: TOK is stage money for demonstrating *settlement mechanics*;
scarcity is not the lesson, and a real deployment deletes this contract entirely.

Contrast with **ERC-721** (§6's `A2ASettlement`): **non-fungible** tokens — each token a
unique id with its own data. TOK answers "how much?"; a ticket answers "which one, and on
what terms?". Money vs deeds.

In [ ]:
print((ROOT / "contracts" / "src" / "MockTOK.sol").read_text())

`w3.eth.contract(address=…, abi=…)` binds the menu to the deployed address — a Python
handle to code living on-chain. Reads go through `.functions.X(...).call()` — free, like
every call. Watch: symbol `TOK`, 18 decimals, and Ada's TOK balance is **zero** — 10,000
ETH, yet 0 TOK. Two moneys, two ledgers, now visible side by side.

In [ ]:
if not CHAIN_OK:
    print(SKIP)
else:
    tok = w3.eth.contract(address=anvil.deployment["MockTOK"], abi=artifacts.load_abi("MockTOK"))
    symbol, decimals = tok.functions.symbol().call(), tok.functions.decimals().call()
    ada_tok = tok.functions.balanceOf(fx.ADA).call()
    print("symbol  →", symbol, "· decimals →", decimals)
    print("Ada TOK →", ada_tok, "  (10,000 ETH, yet 0 TOK — two separate moneys)")
    assert (symbol, decimals, ada_tok) == ("TOK", 18, 0)

Now a **write**: mint Ada 50 TOK through the faucet — a transaction, so it needs Ada's
signature. Hand-building tx dicts gets old; this time we inject web3's signing
**middleware** (the exact pattern `launch_anvil` used — scroll back to §3's source),
after which `w3` signs everything as Ada automatically and `.transact()` just works.

Watch `gasUsed`: ~68,000 vs the plain transfer's 21,000. The faucet *computes more* — it
updates the token's balance mapping in storage — and gas is the meter on computation.
"Fee per computation", made concrete.

In [ ]:
from web3.middleware import SignAndSendRawMiddlewareBuilder

if not CHAIN_OK:
    print(SKIP)
else:
    w3.middleware_onion.inject(SignAndSendRawMiddlewareBuilder.build(ada), layer=0)
    w3.eth.default_account = ada.address                  # w3 signs as Ada from here on

    receipt = w3.eth.wait_for_transaction_receipt(tok.functions.faucet(fx.ADA, 50 * 10**18).transact())

    print("faucet gasUsed →", receipt["gasUsed"], "  (the plain transfer was 21000)")
    print("Ada TOK        →", tok.functions.balanceOf(fx.ADA).call())
    assert receipt["gasUsed"] > 21_000                    # storage writes cost extra computation
    assert tok.functions.balanceOf(fx.ADA).call() == 50 * 10**18
    print("50 TOK minted ✓ — integer money, 18 decimals, exactly like 04")

Break it on purpose: run the very same faucet through `.call()` instead of `.transact()`.
A call may execute *any* function — but as a **simulation**: the node runs the code,
reports what would happen, and throws the result away. Prediction: it returns quietly,
and Bell's balance doesn't move.

In [ ]:
if not CHAIN_OK:
    print(SKIP)
else:
    result = tok.functions.faucet(fx.BELL, 7 * 10**18).call()   # .call(), not .transact()!
    print("faucet .call() returned →", result)
    print("Bell TOK afterwards     →", tok.functions.balanceOf(fx.BELL).call())
    assert tok.functions.balanceOf(fx.BELL).call() == 0
    print("\nnothing minted ✓ — a call SIMULATES; only a transaction changes the world")

**✏️ Your turn 5.1 — fund Bell for real**

Do what the simulation didn't: faucet **10 TOK** to Bell with `.transact()`. Before
running, predict Bell's balance in *base units* (`04`'s integer-money drill: 10 TOK =
10 × 10¹⁸). Success: the assert passes and the printed number is a 1 followed by
nineteen zeros (twenty digits).

In [ ]:
if not CHAIN_OK:
    print(SKIP)
else:
    # TODO — predict Bell's balance in base units BEFORE running: __________
    w3.eth.wait_for_transaction_receipt(tok.functions.faucet(fx.BELL, 10 * 10**18).transact())
    bell_tok = tok.functions.balanceOf(fx.BELL).call()
    print("Bell TOK →", bell_tok)
    # assert bell_tok == 10 * 10**18   # un-comment once you have predicted

<details><summary>✅ Solution 5.1 — peek only after trying</summary>

```python
assert bell_tok == 10 * 10**18   # 10_000_000_000_000_000_000 — a 1 and nineteen zeros
```

`.transact()` made it a real state change this time; 10 TOK in base units is
`10 * 10**18`, the same integer-money convention as `fx.PRICE_10_TOK`.
</details>

## 6. Reading `Settlement.sol` — a Solidity course on the real thing

You have now *used* contracts. Time to *read* one — the one this repo is about:
`contracts/src/Settlement.sol`, 242 lines, the entire marketplace's law. We'll walk it
section by section, printing real excerpts (with real line numbers) through a tiny
helper, and glossing each Solidity construct as it first appears.

Start at the top. Three things live in a Solidity file header:

- `// SPDX-License-Identifier` — a machine-readable license tag (the compiler insists);
- `pragma solidity ^0.8.30` — the compiler-version pin, Solidity's cousin of
  `requires-python`;
- named imports — `import {ERC721} from "@openzeppelin/..."` reads just like Python's
  `from … import`. **OpenZeppelin** is Solidity's audited standard library; these seven
  imports are battle-tested building blocks the contract composes instead of reinventing.

In [ ]:
SETTLEMENT_LINES = (ROOT / "contracts" / "src" / "Settlement.sol").read_text().splitlines()

def sol(first: str, last: str, extra: int = 0) -> None:
    """Print Settlement.sol from the line containing `first` through the line
    containing `last` (+ `extra` lines), with the file's real line numbers."""
    i = next(n for n, line in enumerate(SETTLEMENT_LINES) if first in line)
    j = next(n for n, line in enumerate(SETTLEMENT_LINES[i:], start=i) if last in line) + extra
    for n in range(i, j + 1):
        print(f"{n + 1:3} | {SETTLEMENT_LINES[n]}")

sol("// SPDX-License-Identifier", "import {EIP712}")

Line 25 is the declaration: `contract A2ASettlement is ERC721, EIP712`. A `contract` is
Solidity's class (`01`), and `is` is inheritance — this contract *is an* ERC-721 and *is
an* EIP-712 verifier.

**ERC-721** is the standard for non-fungible tokens (**NFTs**): each token a unique id
with an owner, transferable, queryable (`ownerOf`, `transferFrom`). By inheriting
OpenZeppelin's implementation, the contract gets the entire **ownership ledger** — who
owns which ticket — without writing a line of it, audits included. What this repo adds is
the **terms ledger**: what each ticket *entitles*. Hold that two-ledgers picture; §7
shows it to you physically, slot by slot. (The `EIP712` parent is signature machinery —
`07`'s whole subject.)

In [ ]:
sol("/// A2ASettlement — the on-chain registry", "using SafeERC20")

First construct: **`struct`** — a named record type, Solidity's cousin of `01`'s
dataclass (fields, no validation — the *functions* validate). `Entitlement` is the terms
of one ticket, and its field types show that sizes matter on-chain:

- `address` — 20 bytes (§2's fingerprints, now a *type*);
- `uint8` / `uint64` / `uint256` — unsigned integers of exactly that many bits (smaller
  fields pack tighter into storage, which costs less gas);
- `bytes32` — exactly 32 bytes; `bytes` — a variable-length blob (the `params` you
  decoded by hand in `04`!);
- `bool` — the `revoked` flag this chapter keeps circling back to.

Read the excerpt and count: eight fields. You have met this record before, from the
Python side.

In [ ]:
sol("struct Entitlement", "bytes32 termsHash; // keccak256", extra=1)

**✏️ Your turn 6.1 — spot the extra field**

The struct above has 8 fields; `EntitlementView` (`02`/`04`) is its Python twin. Compute
which field the Python side adds, then explain to yourself where that value lives
on-chain (the `entitlements` mapping two excerpts below is the hint). Success: your set
difference contains exactly one name.

In [ ]:
from a2a_interfaces.models import EntitlementView

STRUCT_FIELDS = {"issuer", "service_type", "resource_id", "params",       # camelCase in
                 "start_time", "end_time", "revoked", "terms_hash"}       # Solidity, snake here
VIEW_FIELDS = set(EntitlementView.model_fields)

only_in_python = None
# TODO: only_in_python = ... (one set minus the other)

print("struct fields:", len(STRUCT_FIELDS), "· view fields:", len(VIEW_FIELDS))
print("only in Python:", only_in_python)
# assert only_in_python == {"id"}   # un-comment once computed

<details><summary>✅ Solution 6.1 — peek only after trying</summary>

```python
only_in_python = VIEW_FIELDS - STRUCT_FIELDS
assert only_in_python == {"id"}
```

On-chain, the id is the **key** of the `entitlements` mapping, not a field of the struct;
the Python view flattens key + value into one record so the controller can pass a single
object around.
</details>

The file's second struct you already know intimately: `Offer` — the twelve fields you
dissected value-by-value in `04`, in the exact order the provider signs them. We only
glance at its doc comment here: *how* those twelve fields become 32 signed bytes (and why
field order is load-bearing) is `07_chainmcp_the_signing_adapter.ipynb`'s whole story.

In [ ]:
sol("/// What a provider signs", "struct Offer {")

Next construct: **`mapping`** — a dict living in contract storage. One crucial difference
from Python's dict: **there is no KeyError**. Every possible key "exists" and reads as
zeros until written. Two mappings carry the whole business:

- `consumed`: digest → bool — the punch-card ledger that makes a signed offer single-use.
  You built this exact idea as Python state in `05` (invariant I2); this is the real one.
- `entitlements`: id → Entitlement — the terms ledger.

And `_lastId`, the id counter. `private` means "no auto-generated getter" — it does
**not** mean secret: all storage is readable off-chain, as §7 will demonstrate. Note the
doc comment's promise, kept two excerpts down: ids count from 1.

In [ ]:
sol("/// digest → already fulfilled", "uint256 private _lastId;")

The no-KeyError rule has a consequence worth a live demonstration — *the existence
gotcha*. Our chain is fresh: nothing is minted (minting is `fulfill`'s job, and driving
`fulfill` is `07`'s). Yet `entitlements(7)` will not fail — it returns an **all-zero
struct**, indistinguishable from "a ticket with zero everything". Asking `ownerOf(7)`, by
contrast, **reverts** — a revert is the EVM (Ethereum Virtual Machine — the engine
every node runs contract code on) aborting the call with an error, which web3
surfaces as a raised exception carrying raw error bytes. The lesson the controller lives
by in `08`: judge existence by `ownerOf`, never by the struct read. Keep the raw hex in
view — we decode it two cells down.

In [ ]:
if not CHAIN_OK:
    print(SKIP)
else:
    stl = w3.eth.contract(address=anvil.deployment["A2ASettlement"],
                          abi=artifacts.load_abi("A2ASettlement"))

    zeros = stl.functions.entitlements(7).call()
    print("entitlements(7) →", zeros)
    assert zeros[0] == "0x" + "0" * 40 and zeros[6] is False   # an all-zero struct, not an error

    owner_err = None
    try:
        stl.functions.ownerOf(7).call()
    except Exception as err:
        owner_err = err
        print("\nownerOf(7)      →", type(err).__name__)
        print("raw error data  →", err.data)
    assert owner_err is not None                                # the revert really happened

Those raw bytes are a **custom error** — Solidity's named failures, the construct `01`'s
exceptions prepared you for. Declared like `error OfferExpired();`, raised like
`revert OfferExpired();`, and on the wire each becomes a compact 4-byte **selector**: the
first four bytes of `keccak256("OfferExpired()")`. Read the five declarations and the
comment above them: three names (`OfferExpired`, `WrongConsumer`, `OfferAlreadyUsed`)
deliberately mirror the `FakeChain` exceptions you triggered in `05` — one spec, two
stacks. Two are new: `BadSignature` (fakes don't verify signatures) and `NotIssuer`
(fakes don't know who's calling).

In [ ]:
sol("// The three shared deny-path names", "error NotIssuer();")

How does Python turn `0x7e273289…` back into a name? `chainmcp.artifacts.error_selectors`
builds the reverse table straight from the ABI — keccak each error's signature string,
keep 4 bytes — so it automatically covers OpenZeppelin's inherited errors too. Watch it
decode §6's mystery revert; every refusal this contract can ever name is on the list
(`07`'s `ChainRevert` exceptions are this table, productionized).

In [ ]:
if not CHAIN_OK:
    print(SKIP)
else:
    table = artifacts.error_selectors(artifacts.load_abi("A2ASettlement"))
    print("every refusal this contract can name:")
    for sel, name in sorted(table.items(), key=lambda kv: kv[1]):
        print(f"  0x{sel.hex()} → {name}")

    selector = bytes.fromhex(owner_err.data[2:10])   # first 4 bytes of the revert data
    print("\nthe mystery revert decodes to →", table[selector])
    assert table[selector] == "ERC721NonexistentToken"

**✏️ Your turn 6.2 — a selector by hand**

Compute the 4-byte selector for `OfferExpired()` yourself — keccak the signature string,
keep four bytes — and look it up in the table. Success: the assert passes; you have
reproduced, by hand, the exact bytes anvil will send back when `07` lets a quote go
stale.

In [ ]:
from eth_utils import keccak

if not ARTIFACTS_OK:
    print(SKIP)
else:
    table = artifacts.error_selectors(artifacts.load_abi("A2ASettlement"))
    selector = None
    # TODO: selector = keccak(text=...)[:4]
    print("selector →", selector if selector is None else "0x" + selector.hex())
    print("lookup   →", None if selector is None else table.get(selector))
    # assert table[selector] == "OfferExpired"   # un-comment once computed

<details><summary>✅ Solution 6.2 — peek only after trying</summary>

```python
selector = keccak(text="OfferExpired()")[:4]
assert table[selector] == "OfferExpired"
```

Hash the *signature string* — the name plus parenthesized argument types, none here —
and keep four bytes. Same recipe for any error or function on any contract.
</details>

Next construct: **`event`** — the chain's log lines. `emit EntitlementMinted(...)`
appends a record to the transaction's receipt; contracts can never read events back —
they exist *for the outside world* to subscribe to. `indexed` parameters become
searchable topics ("all `Revoked` events for id 7", without replaying history).

Read the three declarations and mark `Revoked(uint256 indexed id)`: that one line is what
the controller's revocation machinery will *live on* — `07` watches it fire from Python,
`08` and `12` tear down a live session because of it.

In [ ]:
sol("event EntitlementMinted", "event Revoked(")

Now the centerpiece — read `fulfill` whole, but as an **outline**: running it, the
allowance dance, and the EIP-712 digest that `hashOffer` computes (line 138 — the offer's
32-byte fingerprint) are all `07`'s. What to collect on this pass:

**Four checks, in `05`'s exact order** — you triggered every one of these on cardboard:
stale quote → `OfferExpired`; offer bound to someone else → `WrongConsumer`
(`msg.sender` is Solidity's "who is calling?" — the authentication primitive); digest
already punched → `OfferAlreadyUsed`; signature doesn't recover to the provider →
`BadSignature`.

**Then six effects:** ① punch `consumed[digest]` ② pull the payment (buyer → provider)
③ mint the ticket via `_issue` ④ emit `OfferConsumed` ⑤ emit `EntitlementMinted`
⑥ return the new id. Any failure anywhere reverts *everything* — atomicity is the EVM's
rollback, free of charge (invariant I3; `05` had to hand-roll this with check ordering).

Two subtleties worth their comments: the punch happens *before* the external token call
(effects-before-interactions — a reentrant replay dies at `OfferAlreadyUsed`), and the
service window is **deliberately not checked** — buying at 13:45 for the 14:00 window is
legitimate; *using* it is the controller's decision, against chain time (ADR-004).

In [ ]:
sol("/// The vending machine", "emit EntitlementMinted", extra=1)

Where do tickets actually come from? `_issue` — and its visibility is the security
property: **`internal`** means only this contract (and subclasses) can call it. No public
mint exists; `fulfill` is the single door (invariant I1). And there is the id rule, in
code: `id = ++_lastId` — pre-increment, so the first ticket is **1**, and the story's
"ticket #7" literally means *the seventh ever sold*.

In [ ]:
sol("/// Mint a fresh entitlement", "_mint(to, id);", extra=1)

Don't take I1 on faith — check the menu. Filter the ABI for every function that can
*change state* (anything not `view`/`pure`): six names — inherited transfer/approval
plumbing, plus `fulfill` and `revoke`. Nothing called mint. The harness trick the cast
labs use to reach `_issue` (a test-only subclass) only underlines it: on the real
contract, tickets are bought or they don't exist.

In [ ]:
if not ARTIFACTS_OK:
    print(SKIP)
else:
    abi = artifacts.load_abi("A2ASettlement")
    writes = sorted({e["name"] for e in abi
                     if e.get("type") == "function"
                     and e.get("stateMutability") not in ("view", "pure")})
    print("state-changing functions →", writes)
    assert "fulfill" in writes and "revoke" in writes
    assert not any("mint" in name.lower() for name in writes)
    print("\nno public mint exists — the only door that creates a ticket is fulfill (I1) ✓")

The issuer's kill switch. Read the guard first: `entitlements[id].issuer != msg.sender` —
only the **issuer** (Bell) may revoke; the *owner* (Ada) cannot, because the switch
belongs to the party bound by the promise, not to its beneficiary. The same line doubles
as the existence check: an unminted id has issuer `address(0)`, and `msg.sender` can
never be the zero address.

Then the body: **one bit**, `revoked = true`, plus the event — never a burn. Why keep a
dead ticket? Because a revoked (or expired) ticket must remain **readable evidence** of
what was promised — for disputes, audits, and the measurements in the evaluation chapters
(invariants I5/I8). Re-revoking succeeds on purpose (flag re-set, event re-fired):
teardown is idempotent everywhere (rule 8 — you proved the downstream half in `05`).

And notice what has *no function at all*: expiry. Nothing happens on-chain at `endTime` —
staleness is a judgment each reader makes against chain time. Whoever must *act* on
expiry (the controller) needs its own watcher (`08`).

In [ ]:
sol("/// The issuer's kill switch", "emit Revoked(entitlementId);", extra=1)

Last stop: `tokenURI` — every ERC-721 exposes one, a URI pointing at the token's metadata
(the "fine print"). Normally that's an `https://` URL on someone's server — which can
404, change, or lie. This one is built **from storage, on every call**, and returned as a
`data:application/json;base64,…` URI: the document *is* the ticket's current on-chain
state. No server, nothing to 404 — and it cannot lie: revoke the ticket and the JSON
flips with the flag (invariant I7). `params` is deliberately omitted (an ABI blob isn't
JSON-friendly; readers take it from the `entitlements` getter). Decoding one live is
`07`'s treat, once something is actually minted.

In [ ]:
sol("/// The ticket's fine print", 'return string.concat("data:application/json;base64,', extra=1)

**✏️ Your turn 6.3 — break `tokenURI` and name the error**

Call `tokenURI(1)` on our fresh chain and decode the revert with the selector table.
Predict the error name *before* running — line 190 of the excerpt tells you which
existence rule `tokenURI` follows. Success: the name you predicted is the name the chain
sends back.

In [ ]:
if not CHAIN_OK:
    print(SKIP)
else:
    # TODO — write your predicted error name here: __________
    try:
        stl.functions.tokenURI(1).call()
        print("no error!?")
    except Exception as err:
        name = table.get(bytes.fromhex(err.data[2:10]))
        print("tokenURI(1) reverts with →", name)
        # assert name == "ERC721NonexistentToken"   # un-comment after predicting

<details><summary>✅ Solution 6.3 — peek only after trying</summary>

```python
# prediction: ERC721NonexistentToken — line 190 is `_requireOwned(tokenId)`,
# the same existence rule as ownerOf, so an unminted id reverts identically.
assert name == "ERC721NonexistentToken"
```

`tokenURI` refuses to render fine print for a ticket that was never sold — same rule,
same selector, as the `ownerOf(7)` gotcha above.
</details>

## 7. The two ledgers, physically: storage slots

One more way to see the architecture — the compiler's own map. A contract's storage is a
huge array of numbered **32-byte slots**; each state variable gets one (mappings use
theirs as an anchor and hash their entries elsewhere). Inheritance lays the *parents'*
variables down first. `forge inspect` prints this layout without deploying anything.

So the two-ledgers picture becomes physical. **Slots 0–5** are OpenZeppelin ERC721's —
name, symbol, owners, balances, approvals — the ownership ledger you didn't write.
**Slots 6–7** are OpenZeppelin EIP712's fallback strings (empty here; signature plumbing,
`07`). **Slots 8–10** are the three declarations you read in §6 — `consumed`,
`entitlements`, `_lastId` — the terms ledger this repo actually adds.

In [ ]:
import json
import shutil
import subprocess

FORGE_OK = shutil.which("forge") is not None

def who(slot: int) -> str:
    if slot <= 5:
        return "OpenZeppelin ERC721 — the ownership ledger you didn't write"
    if slot <= 7:
        return "OpenZeppelin EIP712 — signature plumbing (07), empty here"
    return "Settlement.sol — the terms ledger you read today"

if not FORGE_OK:
    print("skipped: needs the forge binary — install Foundry, then rerun this cell")
else:
    out = subprocess.run(
        ["forge", "inspect", "src/Settlement.sol:A2ASettlement", "storageLayout", "--json"],
        cwd=ROOT / "contracts", capture_output=True, text=True, check=True,
    )
    layout = json.loads(out.stdout)["storage"]
    for entry in layout:
        print(f"slot {int(entry['slot']):2}  {entry['label']:<19} ← {who(int(entry['slot']))}")
    slots = {entry["label"]: int(entry["slot"]) for entry in layout}
    assert slots["consumed"] == 8 and slots["entitlements"] == 9 and slots["_lastId"] == 10

**✏️ Your turn 7.1 — MockTOK's layout, predicted**

Run the same inspection on `MockTOK`. Predict first: how many rows will its layout have
(it *is an* `ERC20` and adds nothing of its own — scroll back to §5's reading), and will
`faucet` appear? Success: your predictions match, and you can say *why* `faucet` is
absent in one sentence.

In [ ]:
if not FORGE_OK:
    print("skipped: needs the forge binary — install Foundry, then rerun this cell")
else:
    # TODO — predict BEFORE running:  how many rows? __   does `faucet` appear? __
    out = subprocess.run(
        ["forge", "inspect", "src/MockTOK.sol:MockTOK", "storageLayout", "--json"],
        cwd=ROOT / "contracts", capture_output=True, text=True, check=True,
    )
    rows = json.loads(out.stdout)["storage"]
    print([entry["label"] for entry in rows], "→", len(rows), "rows")
    # assert len(rows) == 5
    # assert "faucet" not in [entry["label"] for entry in rows]

<details><summary>✅ Solution 7.1 — peek only after trying</summary>

```python
assert len(rows) == 5                                    # all five inherited from ERC20
assert "faucet" not in [entry["label"] for entry in rows]
```

Five rows — `_balances`, `_allowances`, `_totalSupply`, `_name`, `_symbol`, every one
inherited from OpenZeppelin's ERC20. `faucet` is absent because **functions are code, not
state** — only variables occupy storage slots.
</details>

## 8. Where this leaves you — and the cast labs

You can now read every line of the marketplace's law and poke its world from Python. Two
loose ends, kept honest:

- **Ids count from 1** (`++_lastId`), and nothing was minted today — minting requires a
  signed offer, an allowance, and `fulfill`, which is exactly `07`'s program. So "ticket
  #7" is a *seventh-issue* fact: in `07`, six sales happen before Ada's so the story
  stays literally true. A fresh chain's first ticket is #1, and the labs say so too.
- Everything you did through web3.py has a terminal twin: **`cast`**, Foundry's chain
  Swiss-army knife (`cast call` = `.call()`, `cast send` = `.transact()`). The repo ships
  four guided **cast labs** under `contracts/EXPLORE*.md` — this chapter's material,
  driven entirely from two terminals. They are your next stop if you want the chain under
  your fingers without Python in between.

A taste, runnable in two terminals at the repo root:

```sh
cd contracts
anvil                                   # terminal 1: a chain on :8545, dev keys preloaded

# terminal 2 — deploy the real way (Deploy.s.sol from §3), then poke storage like §6:
RPC=http://127.0.0.1:8545
forge script script/Deploy.s.sol --rpc-url $RPC --broadcast \
  --private-key 0xac0974bec39a17e36ba4a6b4d238ff944bacb478cbed5efcae784d7bf4f2ff80
STL=$(jq -r .A2ASettlement deployments/anvil.json)

# the existence gotcha, cast edition: an all-zero struct vs a revert
cast call $STL "entitlements(uint256)(address,uint8,bytes32,bytes,uint64,uint64,bool,bytes32)" 7 --rpc-url $RPC
cast call $STL "ownerOf(uint256)(address)" 7 --rpc-url $RPC    # reverts: 0x7e273289…

# name that selector (your-turn 6.2, cast edition)
cast sig "ERC721NonexistentToken(uint256)"                     # → 0x7e273289

# decode the canonical params blob by hand (04's blob, cast edition)
cast abi-decode "f()(uint64,uint8)" 0x0000000000000000000000000000000000000000000000000000000002faf0800000000000000000000000000000000000000000000000000000000000000001
```

Lab order: [`EXPLORE.md`](../../../contracts/EXPLORE.md) (toolchain) →
[`EXPLORE-settlement.md`](../../../contracts/EXPLORE-settlement.md) (storage & ownership) →
[`EXPLORE-fulfill.md`](../../../contracts/EXPLORE-fulfill.md) (the purchase + four cheats) →
[`EXPLORE-revoke.md`](../../../contracts/EXPLORE-revoke.md) (death & afterlife).

One cell of housekeeping left: fold the world away — `anvil.stop()` terminates the
subprocess. Rule of the house: never orphan a chain.

In [ ]:
if anvil is not None:
    anvil.stop()
    print("world folded away — rerun from the top anytime")
else:
    print("no chain was running — nothing to fold away")

## What you learned (and where it goes)

| You did | The concept | Where it goes next |
|---|---|---|
| derived `fx.ADA` from a dev key | an account is a key pair; the address is a fingerprint | offer signing & key custody, EIP-712 (07) |
| read `launch_anvil`, then ran it | disposable worlds; chain time pinned to story time (ADR-004) | every chain notebook; worlds & profiles (11) |
| compared two worlds' `deployment` dicts | addresses = f(deployer, nonce) — deterministic deploys | `anvil.json`, the deployment artifact (11) |
| sent 1 wei and counted blocks | transaction vs call — paid event vs free question | the controller only ever calls (08) |
| fauceted TOK and compared `gasUsed` | a contract = code + storage; gas meters computation | the allowance dance, `approve_and_fulfill` (07) |
| `.call()` on `faucet` changed nothing | calls simulate; only transactions commit | probing deny paths without spending (07) |
| read all 242 lines of `Settlement.sol` | structs, mappings, events, custom errors, inheritance | driving `fulfill`/`revoke`/`tokenURI` live (07) |
| proved no mint exists in the ABI | invariant I1 — `fulfill` is the only door | the purchase, end to end (07) |
| decoded `0x7e273289` by hand | selectors; `error_selectors` is the revert decoder | `ChainRevert(name)` exceptions (07) |
| ran `forge inspect storageLayout` | two ledgers: OZ's ownership + your terms, slot by slot | the ticket as load-bearing evidence (08) |

## Experiments to try

Predict the outcome *before* running each one:

- Re-run §3's launch cell, then §4's first-contact cell. Which asserts still pass
  unchanged, and which printed value differs every time? (Two things vary; one is a
  port.)
- In §4's 1-wei transfer, change `gas` to `20_999` and resend. Read the error carefully —
  is it a contract revert like §6's, or something else entirely? Who rejected it: the
  node's basic rules, or any contract?
- Warp time: `anvil.increase_time(w3, 8 * 3600)` (the labs-only lever from §3's
  dataclass), then re-read `get_block("latest")["timestamp"]`. Is the chain now past
  `fx.WINDOW.end` — and what would that mean for a `fulfill` attempt in `07`?
- Deliberate breakage: run the cleanup cell (`anvil.stop()`), then try any §4 read. Which
  exception type surfaces — and why is "the chain is gone" a *connection* error rather
  than a revert?

## Check yourself

1. Ada "has an account" on this chain. What exactly exists, and where, that makes this
   true? (Hint: nothing was registered anywhere.)
2. Why does the notebook pin anvil's timestamp instead of letting the chain follow your
   laptop's clock? Which ADR is that, and who else will read that clock?
3. A colleague claims "the contract checks that you only buy during the service window."
   Point at the lines that prove them wrong, and explain why that's a feature, not a bug.
4. `entitlements(999)` returns peacefully; `ownerOf(999)` reverts. Which one must the
   controller use to test existence, and what exactly does the other one return?
5. Bell revokes ticket #7 twice. What happens the second time, which repo rule wants it
   that way — and what happens on-chain when the ticket *expires*?

**Where this goes next:** `07_chainmcp_the_signing_adapter.ipynb` — the Python adapter
that drives everything you just read: EIP-712 signing byte-for-byte against `hashOffer`,
the allowance dance, `approve_and_fulfill`, every deny path as a named `ChainRevert`, and
the `Revoked` watcher firing live.

**Deeper dive:** the compact tour
[`chain_client_explore.ipynb`](../chain_client_explore.ipynb) (the full purchase from
Python, including the six pre-sales that make Ada's ticket #7) · the cast labs starting
at [`contracts/EXPLORE.md`](../../../contracts/EXPLORE.md) ·
[`docs/03-interfaces.md`](../../../docs/03-interfaces.md) §1.4/§2 (the shapes and the
domain the signatures pin).